# BT5153 Group Prject Codes

# Topic: Toxic Comment Detection

In [ ]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
# link to Google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# import packages
import os
import re
import json
import random
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from collections import Counter
from datasets import Dataset
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from dataclasses import dataclass
from typing import Dict, List, Optional, Union
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)

In [ ]:
# import datasets
# Update these paths if your files are stored elsewhere.

TOXIC_TRAIN_PATH = "/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/jigsaw-toxic-comment-train.csv"
UNINTENDED_BIAS_TRAIN_PATH = "/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/jigsaw-unintended-bias-train.csv"
VALIDATION_PATH = "/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/validation.csv"
TEST_PATH = "/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/test.csv"

# Optional: if you also have test labels later
TEST_LABELS_PATH = "/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/test_labels.csv"

# Module 5: Threshold Optimization

**Goal**: In a moderation setting, the default 0.5 cutoff is rarely optimal. This module compares **four** threshold-design strategies and reports how each affects recall / precision / FPR / F1 on the multilingual validation set.

**Threshold-selection criterion** (applied to every strategy that tunes): among thresholds achieving `recall ≥ target_recall`, pick the one that **maximizes F1**. Matches proposal Section 2.2.3.

**Strategies**:

| # | Name | Tune on | Eval on | Deployable? |
|---|---|---|---|---|
| A | `default_0.5` | — | full val | trivial baseline |
| B | `global_recall` | English dev | full val | ✅ yes (no leak) |
| C | `per_language_oracle` | each language's val subset | same data | ❌ no (oracle upper bound) |
| D | `per_language_proper` | `val_tune` (50% stratified split) | `val_eval` (other 50%) | ✅ yes (leak-free) |

**Target recalls swept** (Strategy B only): 0.5, 0.7 (primary), 0.9.
**Primary target recall** (used throughout reporting): 0.7.

**Three useful gaps** (reported in Module 5.6):
- `C − B` — theoretical upper bound from per-language calibration (with leak)
- `D − B` — realistic gain from per-language calibration (leak-free, noisier because n is halved)
- `C − D` — estimation noise introduced by the val split

**Outputs** (saved to `module5_outputs/`):
- Strategy A+B: `strategy_ab_thresholds.csv`, `strategy_ab_val_metrics.csv`
- Strategy C:   `strategy_c_per_language_thresholds.csv`, `strategy_c_per_language_metrics.csv`, `strategy_c_overall.csv`
- Strategy D:   `strategy_d_per_language_thresholds.csv`, `strategy_d_per_language_metrics.csv`, `strategy_d_overall.csv`, `val_split_indices.csv`
- Unified:      `strategy_comparison_4way.csv`

**Runtime order**: 5.1 → 5.2 → 5.3 → 5.4 → 5.5 → 5.6 → 5.7 → 5.8.

In [ ]:
# Module 5.1: Utility functions
from sklearn.metrics import average_precision_score


def print_section(title: str) -> None:
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def metrics_at_threshold(y_true, y_prob, thr: float) -> dict:
    """Compute metrics at a given threshold. Includes PR-AUC and FNR."""
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    has_both = len(np.unique(y_true)) > 1
    return {
        "threshold": float(thr),
        "roc_auc":   float(roc_auc_score(y_true, y_prob))           if has_both else np.nan,
        "pr_auc":    float(average_precision_score(y_true, y_prob)) if has_both else np.nan,
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall":    float(recall_score(y_true, y_pred, zero_division=0)),
        "f1":        float(f1_score(y_true, y_pred, zero_division=0)),
        "fpr":       float(fp / (fp + tn)) if (fp + tn) > 0 else 0.0,
        "fnr":       float(fn / (fn + tp)) if (fn + tp) > 0 else 0.0,
        "positive_prediction_rate": float(np.mean(y_pred)),
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
    }


def tune_threshold_for_recall(y_true, y_prob, target_recall: float, grid=None) -> float:
    """
    Proposal-aligned threshold selection (Section 2.2.3):
    Among thresholds achieving recall >= target_recall, return the one maximizing F1.

    Vectorized O(n log n) via cumulative sums on descending-probability sort.
    `grid` kept in signature for backwards compatibility but is ignored.
    """
    y = np.asarray(y_true).astype(int)
    p = np.asarray(y_prob).astype(float)
    n = len(p)
    n_pos = int(y.sum())
    if n_pos == 0 or n_pos == n:
        return 0.5

    order = np.argsort(-p, kind="stable")
    y_sorted = y[order]
    p_sorted = p[order]

    cum_tp = np.cumsum(y_sorted)
    cum_pred_pos = np.arange(1, n + 1)

    recall    = cum_tp / n_pos
    precision = cum_tp / cum_pred_pos
    denom     = precision + recall
    f1        = np.where(denom > 0, 2 * precision * recall / np.maximum(denom, 1e-12), 0.0)

    valid = recall >= target_recall
    if not valid.any():
        return 0.0

    f1_masked = np.where(valid, f1, -np.inf)
    idx = int(np.argmax(f1_masked))
    return float(p_sorted[idx])

In [ ]:
# Module 5.2: Load Module 4 prediction files AND build a stratified val split
print_section("Loading Module 4 predictions")

PROJECT_ROOT = "/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification"
OUTPUT_PATH  = f"{PROJECT_ROOT}/module4_outputs"
MODULE5_PATH = f"{PROJECT_ROOT}/module5_outputs"
os.makedirs(MODULE5_PATH, exist_ok=True)

# Constants used throughout Module 5
TARGET_RECALLS = [0.5, 0.7, 0.9]
PRIMARY_TARGET = 0.7
SPLIT_SEED     = 42

MODELS = {
    "LR":     {"dev_file": "lr_dev_predictions.csv",     "val_file": "lr_validation_predictions.csv",     "prob_col": "pred_prob_lr"},
    "BiLSTM": {"dev_file": "bilstm_dev_predictions.csv", "val_file": "bilstm_validation_predictions.csv", "prob_col": "pred_prob_bilstm"},
    "mBERT":  {"dev_file": "mbert_dev_predictions.csv",  "val_file": "mbert_validation_predictions.csv",  "prob_col": "pred_prob_mbert"},
    "XLM-R":  {"dev_file": "xlmr_dev_predictions.csv",   "val_file": "xlmr_validation_predictions.csv",   "prob_col": "pred_prob_xlmr"},
}

preds = {}
for name, cfg in MODELS.items():
    dev = pd.read_csv(f"{OUTPUT_PATH}/{cfg['dev_file']}").reset_index(drop=True)
    val = pd.read_csv(f"{OUTPUT_PATH}/{cfg['val_file']}").reset_index(drop=True)
    preds[name] = {"dev": dev, "val": val, "prob_col": cfg["prob_col"]}
    print(f"{name:7s} dev={dev.shape}  val={val.shape}  prob_col={cfg['prob_col']}")

for m in MODELS:
    assert "toxic" in preds[m]["dev"].columns, f"{m} dev missing 'toxic'"
    assert {"toxic", "lang"}.issubset(preds[m]["val"].columns), f"{m} val missing 'toxic' or 'lang'"
    assert preds[m]["prob_col"] in preds[m]["dev"].columns
    assert preds[m]["prob_col"] in preds[m]["val"].columns

# Build a stratified 50/50 val split. All 4 models share the same val rows, so one split works.
from sklearn.model_selection import train_test_split

ref_val = preds["LR"]["val"]
strat_key = ref_val["lang"].astype(str) + "__" + ref_val["toxic"].astype(str)

tune_idx, eval_idx = train_test_split(
    ref_val.index.to_numpy(),
    test_size=0.5,
    stratify=strat_key,
    random_state=SPLIT_SEED,
)
tune_idx = sorted(tune_idx.tolist())
eval_idx = sorted(eval_idx.tolist())

print_section("Stratified val split for Strategy D")
print(f"Split seed: {SPLIT_SEED}, stratified by lang x toxic")
print(f"  val_tune: {len(tune_idx)} rows  (used only by Strategy D for threshold tuning)")
print(f"  val_eval: {len(eval_idx)} rows  (used only by Strategy D for reporting)")

tune_dist = ref_val.loc[tune_idx].groupby("lang")["toxic"].agg(n_tune="count", pos_tune="mean")
eval_dist = ref_val.loc[eval_idx].groupby("lang")["toxic"].agg(n_eval="count", pos_eval="mean")
print("\nPer-language sample counts and positive rates in the two halves:")
print(pd.concat([tune_dist, eval_dist], axis=1).round(4).to_string())


Loading Module 4 predictions
LR      dev=(209760, 5)  val=(7999, 6)  prob_col=pred_prob_lr
BiLSTM  dev=(209760, 5)  val=(7999, 6)  prob_col=pred_prob_bilstm
mBERT   dev=(209760, 5)  val=(7999, 6)  prob_col=pred_prob_mbert
XLM-R   dev=(209760, 5)  val=(7999, 6)  prob_col=pred_prob_xlmr

Stratified val split for Strategy D
Split seed: 42, stratified by lang x toxic
  val_tune: 3999 rows  (used only by Strategy D for threshold tuning)
  val_eval: 4000 rows  (used only by Strategy D for reporting)

Per-language sample counts and positive rates in the two halves:
      n_tune  pos_tune  n_eval  pos_eval
lang                                    
es      1250    0.1688    1250    0.1688
it      1250    0.1952    1250    0.1952
tr      1499    0.1067    1500    0.1067


In [ ]:
# Module 5.3: Strategy A (default 0.5) + Strategy B (global, tuned on English dev)
print_section("Strategy A (default 0.5) and Strategy B (global recall-constrained)")

TARGET_RECALLS = [0.5, 0.7, 0.9]
PRIMARY_TARGET = 0.7

threshold_choices = []   # one row per (model, strategy, target_recall)
val_metrics_rows = []    # one row per (model, strategy, target_recall) evaluated on multilingual val

for name, bundle in preds.items():
    dev_df = bundle["dev"]
    val_df = bundle["val"]
    prob_col = bundle["prob_col"]

    y_dev = dev_df["toxic"].values
    p_dev = dev_df[prob_col].values
    y_val = val_df["toxic"].values
    p_val = val_df[prob_col].values

    # Strategy A: default 0.5, evaluated directly on val
    m_val = metrics_at_threshold(y_val, p_val, 0.5)
    threshold_choices.append({
        "model": name, "strategy": "default_0.5", "target_recall": None,
        "scope": "global", "lang": None, "threshold": 0.5,
    })
    val_metrics_rows.append({
        "model": name, "strategy": "default_0.5", "target_recall": None, **m_val,
    })

    # Strategy B: tune on English dev for each target recall, apply to val
    for tr in TARGET_RECALLS:
        thr = tune_threshold_for_recall(y_dev, p_dev, target_recall=tr)
        m_val = metrics_at_threshold(y_val, p_val, thr)
        threshold_choices.append({
            "model": name, "strategy": "global_recall", "target_recall": tr,
            "scope": "global", "lang": None, "threshold": thr,
        })
        val_metrics_rows.append({
            "model": name, "strategy": "global_recall", "target_recall": tr, **m_val,
        })

strategy_ab_thresholds = pd.DataFrame(threshold_choices)
strategy_ab_val_metrics = pd.DataFrame(val_metrics_rows)

print("\n--- Thresholds chosen (A + B) ---")
print(strategy_ab_thresholds.to_string(index=False))

print("\n--- Multilingual val metrics (A + B) ---")
cols = ["model", "strategy", "target_recall", "threshold",
        "precision", "recall", "f1", "fpr", "positive_prediction_rate"]
print(strategy_ab_val_metrics[cols].round(4).to_string(index=False))


Strategy A (default 0.5) and Strategy B (global recall-constrained)

--- Thresholds chosen (A + B) ---
 model      strategy  target_recall  scope lang  threshold
    LR   default_0.5            NaN global None   0.500000
    LR global_recall            0.5 global None   0.777077
    LR global_recall            0.7 global None   0.726207
    LR global_recall            0.9 global None   0.325504
BiLSTM   default_0.5            NaN global None   0.500000
BiLSTM global_recall            0.5 global None   0.854203
BiLSTM global_recall            0.7 global None   0.838730
BiLSTM global_recall            0.9 global None   0.486563
 mBERT   default_0.5            NaN global None   0.500000
 mBERT global_recall            0.5 global None   0.613045
 mBERT global_recall            0.7 global None   0.398706
 mBERT global_recall            0.9 global None   0.007726
 XLM-R   default_0.5            NaN global None   0.500000
 XLM-R global_recall            0.5 global None   0.080902
 XLM-R glob

In [ ]:
# Module 5.4: Strategy C (per-language oracle, target recall = PRIMARY_TARGET)
print_section("Strategy C (per-language oracle)")

per_lang_choices = []
per_lang_val_metrics = []

for name, bundle in preds.items():
    val_df = bundle["val"]
    prob_col = bundle["prob_col"]

    for lang, grp in val_df.groupby("lang"):
        y = grp["toxic"].values
        p = grp[prob_col].values
        if len(np.unique(y)) < 2:
            thr = 0.5  # degenerate group, keep default
        else:
            thr = tune_threshold_for_recall(y, p, target_recall=PRIMARY_TARGET)
        m = metrics_at_threshold(y, p, thr)  # includes pr_auc + fnr
        per_lang_choices.append({
            "model": name, "strategy": "per_language_oracle",
            "target_recall": PRIMARY_TARGET, "scope": "per_language",
            "lang": lang, "threshold": thr, "n": len(grp),
        })
        per_lang_val_metrics.append({
            "model": name, "strategy": "per_language_oracle",
            "target_recall": PRIMARY_TARGET, "lang": lang, "n": len(grp), **m,
        })

strategy_c_thresholds = pd.DataFrame(per_lang_choices)
strategy_c_per_lang_metrics = pd.DataFrame(per_lang_val_metrics)

print("\n--- Per-language thresholds (C) ---")
print(strategy_c_thresholds[["model", "lang", "threshold", "n"]].round(4).to_string(index=False))

# Pooled overall metrics for Strategy C: apply per-language threshold, then aggregate across val.
overall_c_rows = []
for name, bundle in preds.items():
    val_df = bundle["val"].copy()
    prob_col = bundle["prob_col"]
    thr_map = dict(
        strategy_c_thresholds[strategy_c_thresholds["model"] == name]
        .set_index("lang")["threshold"]
    )
    val_df["_thr"]  = val_df["lang"].map(thr_map)
    val_df["_pred"] = (val_df[prob_col] >= val_df["_thr"]).astype(int)

    y      = val_df["toxic"].values
    y_pred = val_df["_pred"].values
    p      = val_df[prob_col].values
    tn, fp, fn, tp = confusion_matrix(y, y_pred, labels=[0, 1]).ravel()
    overall_c_rows.append({
        "model": name, "strategy": "per_language_oracle", "target_recall": PRIMARY_TARGET,
        "threshold": np.nan,  # not a single scalar
        "roc_auc":   float(roc_auc_score(y, p)),
        "pr_auc":    float(average_precision_score(y, p)),
        "precision": float(precision_score(y, y_pred, zero_division=0)),
        "recall":    float(recall_score(y, y_pred, zero_division=0)),
        "f1":        float(f1_score(y, y_pred, zero_division=0)),
        "fpr":       float(fp / (fp + tn)) if (fp + tn) > 0 else 0.0,
        "fnr":       float(fn / (fn + tp)) if (fn + tp) > 0 else 0.0,
        "positive_prediction_rate": float(np.mean(y_pred)),
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
    })

strategy_c_overall = pd.DataFrame(overall_c_rows)

print("\n--- Strategy C pooled overall on multilingual val ---")
cols = ["model", "roc_auc", "pr_auc", "precision", "recall", "f1", "fpr", "fnr", "positive_prediction_rate"]
print(strategy_c_overall[cols].round(4).to_string(index=False))


Strategy C (per-language oracle)

--- Per-language thresholds (C) ---
 model lang  threshold    n
    LR   es     0.0740 2500
    LR   it     0.0540 2500
    LR   tr     0.0558 2999
BiLSTM   es     0.0411 2500
BiLSTM   it     0.0012 2500
BiLSTM   tr     0.0042 2999
 mBERT   es     0.0319 2500
 mBERT   it     0.0115 2500
 mBERT   tr     0.0126 2999
 XLM-R   es     0.0276 2500
 XLM-R   it     0.0182 2500
 XLM-R   tr     0.0170 2999

--- Strategy C pooled overall on multilingual val ---
 model  roc_auc  pr_auc  precision  recall     f1    fpr    fnr  positive_prediction_rate
    LR   0.6223  0.2459     0.1926  0.7642 0.3076 0.5822 0.2358                    0.6102
BiLSTM   0.5679  0.2375     0.1633  0.9187 0.2774 0.8551 0.0813                    0.8649
 mBERT   0.7259  0.3242     0.2680  0.7089 0.3889 0.3519 0.2911                    0.4068
 XLM-R   0.8363  0.4341     0.4118  0.7236 0.5249 0.1878 0.2764                    0.2702


In [ ]:
# Module 5.5: Strategy D -- Proper per-language tuning (tune on val_tune, evaluate on val_eval)
from sklearn.metrics import average_precision_score

print_section("Strategy D: per-language thresholds tuned on val_tune, evaluated on val_eval")


def _metrics_from_predictions(y_true, y_prob, y_pred):
    """Metrics from pre-computed y_pred (per-language pooled eval uses this)."""
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = np.asarray(y_pred).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    has_both = len(np.unique(y_true)) > 1
    n = int(len(y_true))
    return {
        "n": n, "n_pos": int(tp + fn),
        "positive_ratio": float((tp + fn) / n) if n > 0 else 0.0,
        "roc_auc":   float(roc_auc_score(y_true, y_prob))           if has_both else np.nan,
        "pr_auc":    float(average_precision_score(y_true, y_prob)) if has_both else np.nan,
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall":    float(recall_score(y_true, y_pred, zero_division=0)),
        "f1":        float(f1_score(y_true, y_pred, zero_division=0)),
        "fpr":       float(fp / (fp + tn)) if (fp + tn) > 0 else 0.0,
        "fnr":       float(fn / (fn + tp)) if (fn + tp) > 0 else 0.0,
        "positive_prediction_rate": float(np.mean(y_pred)),
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
    }


# Step 1: tune per-language thresholds on val_tune
d_threshold_rows = []
for name, bundle in preds.items():
    val = bundle["val"]
    prob_col = bundle["prob_col"]
    tune_df = val.loc[tune_idx]

    for lang, grp in tune_df.groupby("lang"):
        y = grp["toxic"].values
        p = grp[prob_col].values
        if len(np.unique(y)) < 2:
            thr, note = 0.5, "degenerate"
        else:
            thr = tune_threshold_for_recall(y, p, target_recall=PRIMARY_TARGET)
            note = "ok"
        d_threshold_rows.append({
            "model": name, "strategy": "per_language_proper",
            "target_recall": PRIMARY_TARGET, "scope": "per_language",
            "lang": lang, "threshold": float(thr),
            "n_tune": int(len(grp)), "note": note,
        })
strategy_d_thresholds = pd.DataFrame(d_threshold_rows)

print("--- Per-language thresholds (D, tuned on val_tune) ---")
print(strategy_d_thresholds[["model", "lang", "threshold", "n_tune", "note"]].round(4).to_string(index=False))

# Step 2: evaluate on val_eval (per-language + pooled overall)
d_per_lang_rows = []
d_overall_rows  = []
for name, bundle in preds.items():
    val = bundle["val"]
    prob_col = bundle["prob_col"]
    eval_df  = val.loc[eval_idx].copy()

    thr_map = dict(
        strategy_d_thresholds[strategy_d_thresholds["model"] == name]
        .set_index("lang")["threshold"]
    )

    # Per-language on val_eval
    for lang, grp in eval_df.groupby("lang"):
        y = grp["toxic"].values
        p = grp[prob_col].values
        thr = float(thr_map.get(lang, 0.5))
        y_pred = (p >= thr).astype(int)
        m = _metrics_from_predictions(y, p, y_pred)
        d_per_lang_rows.append({
            "model": name, "strategy": "per_language_proper", "lang": lang,
            "target_recall": PRIMARY_TARGET, "threshold": thr, **m,
        })

    # Pooled overall on val_eval (each row uses its language's threshold)
    eval_df["_thr"]  = eval_df["lang"].map(thr_map)
    eval_df["_pred"] = (eval_df[prob_col] >= eval_df["_thr"]).astype(int)
    y = eval_df["toxic"].values
    p = eval_df[prob_col].values
    y_pred = eval_df["_pred"].values
    m = _metrics_from_predictions(y, p, y_pred)
    d_overall_rows.append({
        "model": name, "strategy": "per_language_proper",
        "target_recall": PRIMARY_TARGET, "threshold": np.nan, **m,
    })

strategy_d_per_lang_metrics = pd.DataFrame(d_per_lang_rows)
strategy_d_overall          = pd.DataFrame(d_overall_rows)

print("\n--- Strategy D per-language metrics on val_eval ---")
cols = ["model", "lang", "threshold", "precision", "recall", "f1", "fpr", "positive_prediction_rate", "n"]
print(strategy_d_per_lang_metrics[cols].round(4).to_string(index=False))

print("\n--- Strategy D pooled overall on val_eval ---")
cols = ["model", "precision", "recall", "f1", "fpr", "fnr", "positive_prediction_rate", "n"]
print(strategy_d_overall[cols].round(4).to_string(index=False))


Strategy D: per-language thresholds tuned on val_tune, evaluated on val_eval
--- Per-language thresholds (D, tuned on val_tune) ---
 model lang  threshold  n_tune note
    LR   es     0.1172    1250   ok
    LR   it     0.0513    1250   ok
    LR   tr     0.0446    1499   ok
BiLSTM   es     0.0394    1250   ok
BiLSTM   it     0.0013    1250   ok
BiLSTM   tr     0.0049    1499   ok
 mBERT   es     0.0324    1250   ok
 mBERT   it     0.0103    1250   ok
 mBERT   tr     0.0111    1499   ok
 XLM-R   es     0.0278    1250   ok
 XLM-R   it     0.0174    1250   ok
 XLM-R   tr     0.0170    1499   ok

--- Strategy D per-language metrics on val_eval ---
 model lang  threshold  precision  recall     f1    fpr  positive_prediction_rate    n
    LR   es     0.1172     0.1838  0.6777 0.2892 0.6112                    0.6224 1250
    LR   it     0.0513     0.2351  0.7090 0.3531 0.5596                    0.5888 1250
    LR   tr     0.0446     0.1482  0.8125 0.2507 0.5575                    0.5847 150

In [ ]:
# Module 5.6: Unified 4-strategy comparison + uplift + gap analysis
print_section("Unified 4-strategy comparison (primary recall target = 0.7)")

primary_ab = strategy_ab_val_metrics[
    (strategy_ab_val_metrics["strategy"] == "default_0.5") |
    ((strategy_ab_val_metrics["strategy"] == "global_recall") &
     (strategy_ab_val_metrics["target_recall"] == PRIMARY_TARGET))
].copy()

unified_4 = pd.concat(
    [primary_ab, strategy_c_overall, strategy_d_overall],
    ignore_index=True, sort=False,
)

cols = ["model", "strategy", "target_recall", "threshold",
        "precision", "recall", "f1", "fpr", "positive_prediction_rate"]
unified_4 = unified_4[[c for c in cols if c in unified_4.columns]] \
            .sort_values(["model", "strategy"]).reset_index(drop=True)

print("Note: A/B/C evaluated on full val; D evaluated on val_eval (50% of val).")
print("Strategy differences dominate any sample-size asymmetry.\n")
print(unified_4.round(4).to_string(index=False))

# Uplift vs default_0.5
print_section("Uplift vs default_0.5 (delta precision, delta recall, delta fpr)")
base = (unified_4[unified_4["strategy"] == "default_0.5"]
        .set_index("model")[["precision", "recall", "fpr"]])
uplift_rows = []
for _, row in unified_4.iterrows():
    if row["strategy"] == "default_0.5":
        continue
    m = row["model"]
    uplift_rows.append({
        "model": m, "strategy": row["strategy"],
        "d_precision": round(row["precision"] - base.loc[m, "precision"], 4),
        "d_recall":    round(row["recall"]    - base.loc[m, "recall"],    4),
        "d_fpr":       round(row["fpr"]       - base.loc[m, "fpr"],       4),
    })
print(pd.DataFrame(uplift_rows).to_string(index=False))

# Gap analysis: C - B (upper bound), D - B (realistic), C - D (noise)
print_section("Per-language calibration value: gap analysis")
print("dF1_D_minus_B  = realistic gain from per-language calibration (leak-free)")
print("dF1_C_minus_B  = theoretical upper bound (oracle uses val labels for tuning)")
print("dF1_C_minus_D  = estimation noise introduced by the val split\n")


def _row(df, model, strategy):
    m = df[(df["model"] == model) & (df["strategy"] == strategy)]
    return m.iloc[0] if len(m) else None


gap_rows = []
for m in ["LR", "BiLSTM", "mBERT", "XLM-R"]:
    b = _row(unified_4, m, "global_recall")
    c = _row(unified_4, m, "per_language_oracle")
    d = _row(unified_4, m, "per_language_proper")
    if b is None or c is None or d is None:
        continue
    gap_rows.append({
        "model": m,
        "dF1_D_minus_B":  round(d["f1"]     - b["f1"],     4),
        "dF1_C_minus_B":  round(c["f1"]     - b["f1"],     4),
        "dF1_C_minus_D":  round(c["f1"]     - d["f1"],     4),
        "dRec_D_minus_B": round(d["recall"] - b["recall"], 4),
        "dFPR_D_minus_B": round(d["fpr"]    - b["fpr"],    4),
    })
print(pd.DataFrame(gap_rows).to_string(index=False))


Unified 4-strategy comparison (primary recall target = 0.7)
Note: A/B/C evaluated on full val; D evaluated on val_eval (50% of val).
Strategy differences dominate any sample-size asymmetry.

 model            strategy  target_recall  threshold  precision  recall     f1    fpr  positive_prediction_rate
BiLSTM         default_0.5            NaN     0.5000     0.2587  0.2057 0.2292 0.1071                    0.1223
BiLSTM       global_recall            0.7     0.8387     0.3942  0.0772 0.1292 0.0216                    0.0301
BiLSTM per_language_oracle            0.7        NaN     0.1633  0.9187 0.2774 0.8551                    0.8649
BiLSTM per_language_proper            0.7        NaN     0.1624  0.9154 0.2758 0.8579                    0.8668
    LR         default_0.5            NaN     0.5000     0.3924  0.0756 0.1268 0.0213                    0.0296
    LR       global_recall            0.7     0.7262     0.5096  0.0431 0.0795 0.0075                    0.0130
    LR per_language_orac

In [ ]:
# Module 5.7: Save all Module 5 outputs (A + B + C + D + unified + val split)
print_section("Saving Module 5 outputs")

# Strategy A + B
strategy_ab_thresholds.to_csv(f"{MODULE5_PATH}/strategy_ab_thresholds.csv", index=False)
strategy_ab_val_metrics.to_csv(f"{MODULE5_PATH}/strategy_ab_val_metrics.csv", index=False)

# Strategy C
strategy_c_thresholds.to_csv(f"{MODULE5_PATH}/strategy_c_per_language_thresholds.csv", index=False)
strategy_c_per_lang_metrics.to_csv(f"{MODULE5_PATH}/strategy_c_per_language_metrics.csv", index=False)
strategy_c_overall.to_csv(f"{MODULE5_PATH}/strategy_c_overall.csv", index=False)

# Strategy D
strategy_d_thresholds.to_csv(f"{MODULE5_PATH}/strategy_d_per_language_thresholds.csv", index=False)
strategy_d_per_lang_metrics.to_csv(f"{MODULE5_PATH}/strategy_d_per_language_metrics.csv", index=False)
strategy_d_overall.to_csv(f"{MODULE5_PATH}/strategy_d_overall.csv", index=False)

# Unified
unified_4.to_csv(f"{MODULE5_PATH}/strategy_comparison_4way.csv", index=False)

# Val split indices for reproducibility
split_info = pd.DataFrame({
    "row_index": list(ref_val.index),
    "lang":      ref_val["lang"].values,
    "toxic":     ref_val["toxic"].values,
    "split":     ["tune" if i in set(tune_idx) else "eval" for i in ref_val.index],
})
split_info.to_csv(f"{MODULE5_PATH}/val_split_indices.csv", index=False)

print(f"Saved to: {MODULE5_PATH}")
for f in sorted(os.listdir(MODULE5_PATH)):
    print(" -", f)

In [ ]:
# Module 5.8: Update project state
STATE_FILE = f"{PROJECT_ROOT}/project_state/run_state.json"

with open(STATE_FILE, "r") as f:
    state = json.load(f)

state["module_5_threshold_optimization"] = {
    "status": "done",
    "target_recalls": TARGET_RECALLS,
    "primary_target_recall": PRIMARY_TARGET,
    "strategies": ["default_0.5", "global_recall", "per_language_oracle", "per_language_proper"],
    "threshold_criterion": "max F1 subject to recall >= target (proposal Section 2.2.3)",
    "val_split": {
        "ratio": "50/50",
        "stratify_by": "lang x toxic",
        "seed": SPLIT_SEED,
    },
    "output_dir": MODULE5_PATH,
    "next_step": "module_6_evaluation",
}

with open(STATE_FILE, "w") as f:
    json.dump(state, f, indent=4)

print("Module 5 marked done in run_state.json.")
print("Next step:", state["module_5_threshold_optimization"]["next_step"])

# Module 6: Evaluation

Consolidated evaluation per proposal:

**Probability-level** (threshold-independent):
- ROC-AUC (overall, per language)
- PR-AUC (overall, per language) — optional in proposal, included here

**Threshold-level** (at four operating points):
- Precision, Recall, F1
- FPR, FNR — optional in proposal, included here
- Positive prediction rate, confusion-matrix counts (for auditability)

**Four threshold settings evaluated**:
1. `default_0.5` — no tuning (reference)
2. `global_recall@0.7` — single per-model threshold tuned on English dev via the proposal's
   criterion (max F1 subject to recall ≥ 0.7); deployable, evaluated on full val
3. `per_language_oracle` — per-(model, language) threshold tuned on val; oracle upper bound,
   evaluated on full val
4. `per_language_proper` — per-(model, language) threshold tuned on `val_tune` (50% stratified
   split); deployable and leak-free, **evaluated only on `val_eval`** (the other 50%)

**Note on sample sizes**: settings 1–3 are evaluated on the full multilingual val (~8k rows);
setting 4 is evaluated on `val_eval` only (~4k rows). The asymmetry is intentional and noted
in printouts; strategy differences dominate sample-size effects.

**Prerequisite**: Module 5 must have run at least through its in-memory DataFrames.
Module 6 prefers to read Module 5's saved CSVs from `module5_outputs/`; if those
do not exist, it falls back to the in-memory variables.

**Outputs** (saved to `module6_outputs/`):
- `module6_overall_metrics.csv` — 4 models × 4 settings on multilingual val / val_eval
- `module6_per_language_metrics.csv` — 4 models × languages × 4 settings
- `module6_roc_auc_pivot.csv`, `module6_pr_auc_pivot.csv`, `module6_f1_pivot.csv` — pivoted views
- `module6_per_language_f1_*.csv` — per-language F1 under each tuning setting

In [ ]:
# Module 6.1: Utilities + load predictions + load Module 5 threshold choices
from sklearn.metrics import average_precision_score

def full_metrics(y_true, y_prob, thr: float) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    has_both = len(np.unique(y_true)) > 1
    n = int(len(y_true))
    return {
        "threshold": float(thr),
        "n":         n,
        "n_pos":     int(tp + fn),
        "positive_ratio": float((tp + fn) / n) if n > 0 else 0.0,
        "roc_auc":   float(roc_auc_score(y_true, y_prob))           if has_both else np.nan,
        "pr_auc":    float(average_precision_score(y_true, y_prob)) if has_both else np.nan,
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall":    float(recall_score(y_true, y_pred, zero_division=0)),
        "f1":        float(f1_score(y_true, y_pred, zero_division=0)),
        "fpr":       float(fp / (fp + tn)) if (fp + tn) > 0 else 0.0,
        "fnr":       float(fn / (fn + tp)) if (fn + tp) > 0 else 0.0,
        "positive_prediction_rate": float(np.mean(y_pred)),
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
    }


def metrics_from_predictions(y_true, y_prob, y_pred) -> dict:
    """Same as full_metrics but accepts precomputed y_pred (for per-language oracle)."""
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = np.asarray(y_pred).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    has_both = len(np.unique(y_true)) > 1
    n = int(len(y_true))
    return {
        "threshold": np.nan,  # not a single scalar
        "n":         n,
        "n_pos":     int(tp + fn),
        "positive_ratio": float((tp + fn) / n) if n > 0 else 0.0,
        "roc_auc":   float(roc_auc_score(y_true, y_prob))           if has_both else np.nan,
        "pr_auc":    float(average_precision_score(y_true, y_prob)) if has_both else np.nan,
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall":    float(recall_score(y_true, y_pred, zero_division=0)),
        "f1":        float(f1_score(y_true, y_pred, zero_division=0)),
        "fpr":       float(fp / (fp + tn)) if (fp + tn) > 0 else 0.0,
        "fnr":       float(fn / (fn + tp)) if (fn + tp) > 0 else 0.0,
        "positive_prediction_rate": float(np.mean(y_pred)),
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
    }


# --- Paths ---
PROJECT_ROOT = "/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification"
OUTPUT_PATH  = f"{PROJECT_ROOT}/module4_outputs"
MODULE5_PATH = f"{PROJECT_ROOT}/module5_outputs"
MODULE6_PATH = f"{PROJECT_ROOT}/module6_outputs"
os.makedirs(MODULE6_PATH, exist_ok=True)

# --- Reload predictions (defensive: don't rely on in-memory `preds`) ---
print_section("Loading Module 4 predictions")

MODELS = {
    "LR":     {"val_file": "lr_validation_predictions.csv",     "prob_col": "pred_prob_lr"},
    "BiLSTM": {"val_file": "bilstm_validation_predictions.csv", "prob_col": "pred_prob_bilstm"},
    "mBERT":  {"val_file": "mbert_validation_predictions.csv",  "prob_col": "pred_prob_mbert"},
    "XLM-R":  {"val_file": "xlmr_validation_predictions.csv",   "prob_col": "pred_prob_xlmr"},
}

val_dfs = {}
for name, cfg in MODELS.items():
    df = pd.read_csv(f"{OUTPUT_PATH}/{cfg['val_file']}")
    assert {"toxic", "lang", cfg["prob_col"]}.issubset(df.columns), f"{name} val missing required columns"
    val_dfs[name] = df
    print(f"{name:7s} val shape={df.shape}")

# --- Load Module 5 threshold choices (CSV preferred, fall back to in-memory) ---
print_section("Loading Module 5 threshold choices")

try:
    ab_thr = pd.read_csv(f"{MODULE5_PATH}/strategy_ab_thresholds.csv")
    c_thr  = pd.read_csv(f"{MODULE5_PATH}/strategy_c_per_language_thresholds.csv")
    print("Loaded Module 5 thresholds from saved CSVs.")
except FileNotFoundError:
    ab_thr = strategy_ab_thresholds.copy()     # from Module 5 in-memory
    c_thr  = strategy_c_thresholds.copy()
    print("CSVs not found; using in-memory Module 5 threshold DataFrames.")

PRIMARY_TARGET = 0.7

global_thr = {}
for name in MODELS:
    row = ab_thr[
        (ab_thr["model"] == name)
        & (ab_thr["strategy"] == "global_recall")
        & (ab_thr["target_recall"] == PRIMARY_TARGET)
    ]
    assert len(row) == 1, f"Expected one global_recall@{PRIMARY_TARGET} row for {name}, found {len(row)}"
    global_thr[name] = float(row["threshold"].iloc[0])

per_lang_thr = {}
for name in MODELS:
    sub = c_thr[c_thr["model"] == name]
    per_lang_thr[name] = dict(zip(sub["lang"], sub["threshold"].astype(float)))

print("\n--- global_recall@0.7 thresholds ---")
for k, v in global_thr.items():
    print(f"  {k:7s} {v:.4f}")
print("\n--- per-language thresholds (Strategy C) ---")
for name in MODELS:
    thr_str = ", ".join(f"{l}={t:.3f}" for l, t in sorted(per_lang_thr[name].items()))
    print(f"  {name:7s} {thr_str}")

# --- Load Strategy D thresholds + val split (for evaluation on val_eval) ---
try:
    d_thr = pd.read_csv(f"{MODULE5_PATH}/strategy_d_per_language_thresholds.csv")
    split_info = pd.read_csv(f"{MODULE5_PATH}/val_split_indices.csv")
    print("Loaded Strategy D thresholds and val split from CSVs.")
except FileNotFoundError:
    d_thr = strategy_d_thresholds.copy()
    split_info = pd.DataFrame({
        "row_index": list(ref_val.index),
        "split": ["tune" if i in set(tune_idx) else "eval" for i in ref_val.index],
    })
    print("CSVs not found; using in-memory Strategy D + val split.")

eval_idx = sorted(split_info[split_info["split"] == "eval"]["row_index"].tolist())
per_lang_thr_d = {
    name: dict(zip(
        d_thr[d_thr["model"] == name]["lang"],
        d_thr[d_thr["model"] == name]["threshold"].astype(float),
    ))
    for name in MODELS
}
print(f"\nval_eval size: {len(eval_idx)} rows (used by per_language_proper)")

In [ ]:
# Module 6.2: Overall multilingual-val metrics (4 models × 4 settings = 16 rows)
print_section("Module 6.2: Overall metrics on multilingual validation")

overall_rows = []

for name, df in val_dfs.items():
    y = df["toxic"].values
    p = df[MODELS[name]["prob_col"]].values

    # Setting 1: default 0.5 (full val)
    m = full_metrics(y, p, 0.5)
    overall_rows.append({"model": name, "setting": "default_0.5", **m})

    # Setting 2: global_recall@0.7 (full val)
    m = full_metrics(y, p, global_thr[name])
    overall_rows.append({"model": name, "setting": "global_recall@0.7", **m})

    # Setting 3: per-language oracle (full val)
    thr_series = df["lang"].map(per_lang_thr[name])
    y_pred = (p >= thr_series.values).astype(int)
    m = metrics_from_predictions(y, p, y_pred)
    overall_rows.append({"model": name, "setting": "per_language_oracle", **m})

    # Setting 4: per-language proper (val_eval only)
    eval_df = df.loc[eval_idx]
    y_e = eval_df["toxic"].values
    p_e = eval_df[MODELS[name]["prob_col"]].values
    thr_series_e = eval_df["lang"].map(per_lang_thr_d[name])
    y_pred_e = (p_e >= thr_series_e.values).astype(int)
    m = metrics_from_predictions(y_e, p_e, y_pred_e)
    overall_rows.append({"model": name, "setting": "per_language_proper", **m})

overall_df = pd.DataFrame(overall_rows)

cols_order = [
    "model", "setting", "threshold",
    "roc_auc", "pr_auc",
    "precision", "recall", "f1",
    "fpr", "fnr",
    "positive_prediction_rate",
    "tp", "fp", "fn", "tn",
    "n", "n_pos", "positive_ratio",
]
overall_df = overall_df[cols_order]

print("Note: settings 1–3 on full val (~8k); setting 4 on val_eval (~4k).\n")
print(overall_df.round(4).to_string(index=False))

# Pivot views for quick reading
print_section("ROC-AUC pivot (model × setting)")
roc_pivot = overall_df.pivot_table(index="model", columns="setting", values="roc_auc").round(4)
print(roc_pivot)

print_section("PR-AUC pivot")
pr_pivot = overall_df.pivot_table(index="model", columns="setting", values="pr_auc").round(4)
print(pr_pivot)

print_section("F1 pivot")
f1_pivot = overall_df.pivot_table(index="model", columns="setting", values="f1").round(4)
print(f1_pivot)

In [ ]:
# Module 6.3: Per-language multilingual-val metrics (4 models × languages × 4 settings)
print_section("Module 6.3: Per-language metrics on multilingual validation")

per_lang_rows = []

# Settings 1-3 evaluated on full val
for name, df in val_dfs.items():
    prob_col = MODELS[name]["prob_col"]
    for lang, grp in df.groupby("lang"):
        y = grp["toxic"].values
        p = grp[prob_col].values
        for setting_name, thr in [
            ("default_0.5",         0.5),
            ("global_recall@0.7",   global_thr[name]),
            ("per_language_oracle", per_lang_thr[name].get(lang, 0.5)),
        ]:
            m = full_metrics(y, p, thr)
            per_lang_rows.append({"model": name, "lang": lang, "setting": setting_name, **m})

# Setting 4 (per_language_proper) evaluated only on val_eval
for name, df in val_dfs.items():
    prob_col = MODELS[name]["prob_col"]
    eval_df = df.loc[eval_idx]
    for lang, grp in eval_df.groupby("lang"):
        y = grp["toxic"].values
        p = grp[prob_col].values
        thr = per_lang_thr_d[name].get(lang, 0.5)
        m = full_metrics(y, p, thr)
        per_lang_rows.append({"model": name, "lang": lang, "setting": "per_language_proper", **m})

per_lang_df = pd.DataFrame(per_lang_rows)

cols_order = [
    "model", "lang", "setting", "threshold",
    "roc_auc", "pr_auc",
    "precision", "recall", "f1",
    "fpr", "fnr",
    "positive_prediction_rate",
    "tp", "fp", "fn", "tn",
    "n", "n_pos", "positive_ratio",
]
per_lang_df = per_lang_df[cols_order]

print_section("Per-language ROC-AUC (threshold-independent)")
lang_roc = per_lang_df[per_lang_df["setting"] == "default_0.5"].pivot_table(
    index="model", columns="lang", values="roc_auc"
).round(4)
print(lang_roc)

print_section("Per-language F1 under global_recall@0.7")
lang_f1_global = per_lang_df[per_lang_df["setting"] == "global_recall@0.7"].pivot_table(
    index="model", columns="lang", values="f1"
).round(4)
print(lang_f1_global)

print_section("Per-language F1 under per_language_oracle")
lang_f1_oracle = per_lang_df[per_lang_df["setting"] == "per_language_oracle"].pivot_table(
    index="model", columns="lang", values="f1"
).round(4)
print(lang_f1_oracle)

print_section("Per-language F1 under per_language_proper (D, on val_eval)")
lang_f1_proper = per_lang_df[per_lang_df["setting"] == "per_language_proper"].pivot_table(
    index="model", columns="lang", values="f1"
).round(4)
print(lang_f1_proper)

In [ ]:
# Module 6.4: Save outputs
print_section("Saving Module 6 outputs")

overall_df.to_csv(f"{MODULE6_PATH}/module6_overall_metrics.csv", index=False)
per_lang_df.to_csv(f"{MODULE6_PATH}/module6_per_language_metrics.csv", index=False)

roc_pivot.to_csv(f"{MODULE6_PATH}/module6_roc_auc_pivot.csv")
pr_pivot.to_csv(f"{MODULE6_PATH}/module6_pr_auc_pivot.csv")
f1_pivot.to_csv(f"{MODULE6_PATH}/module6_f1_pivot.csv")
lang_roc.to_csv(f"{MODULE6_PATH}/module6_per_language_roc_auc.csv")
lang_f1_global.to_csv(f"{MODULE6_PATH}/module6_per_language_f1_global.csv")
lang_f1_oracle.to_csv(f"{MODULE6_PATH}/module6_per_language_f1_oracle.csv")
lang_f1_proper.to_csv(f"{MODULE6_PATH}/module6_per_language_f1_proper.csv")

print(f"Saved to: {MODULE6_PATH}")
for f in sorted(os.listdir(MODULE6_PATH)):
    print(" -", f)

In [ ]:
# Module 6.6: Generate report figures (saved to module6_outputs/figures/)
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import seaborn as sns

FIG_DIR = f"{MODULE6_PATH}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

MODEL_ORDER   = ["LR", "BiLSTM", "mBERT", "XLM-R"]
SETTING_ORDER = ["default_0.5", "global_recall@0.7", "per_language_oracle", "per_language_proper"]
SETTING_LABEL = {
    "default_0.5":         "Default 0.5",
    "global_recall@0.7":   "Global (dev-tuned)",
    "per_language_oracle": "Per-lang Oracle",
    "per_language_proper": "Per-lang Proper",
}
sns.set_theme(style="whitegrid", context="notebook")
print_section("Module 6.6: Generating report figures")


# ------------------------------------------------------------------
# Figure 1: ROC-AUC overall + per-language (grouped bars by model)
# ------------------------------------------------------------------
overall_roc = (overall_df[overall_df["setting"] == "default_0.5"]
               .set_index("model")["roc_auc"])
lang_roc = (per_lang_df[per_lang_df["setting"] == "default_0.5"]
            .pivot_table(index="model", columns="lang", values="roc_auc"))
plot_df = lang_roc.copy()
plot_df.insert(0, "Overall", overall_roc)
plot_df = plot_df.reindex(MODEL_ORDER)

fig, ax = plt.subplots(figsize=(10, 5))
plot_df.plot.bar(ax=ax, width=0.8, edgecolor="black")
ax.set_title("ROC-AUC by Model (Overall and Per Language)", fontsize=13, fontweight="bold")
ax.set_ylabel("ROC-AUC")
ax.set_xlabel("")
ax.set_ylim(0.5, 1.0)
ax.axhline(0.5, color="grey", linestyle="--", alpha=0.5, linewidth=1)
ax.legend(title="Subset", loc="upper left", bbox_to_anchor=(1.02, 1.0))
plt.xticks(rotation=0)
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=2, fontsize=8)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig1_roc_auc_per_language.png", dpi=150, bbox_inches="tight")
plt.show()


# ------------------------------------------------------------------
# Figure 2: F1 across 4 strategies (grouped bars by model)
# ------------------------------------------------------------------
f1_matrix = (overall_df.pivot_table(index="model", columns="setting", values="f1")
             .reindex(MODEL_ORDER)[SETTING_ORDER])
f1_matrix.columns = [SETTING_LABEL[c] for c in f1_matrix.columns]

fig, ax = plt.subplots(figsize=(10, 5))
f1_matrix.plot.bar(ax=ax, width=0.8, edgecolor="black", colormap="viridis")
ax.set_title("F1 by Model and Threshold Strategy", fontsize=13, fontweight="bold")
ax.set_ylabel("F1 (multilingual val)")
ax.set_xlabel("")
ax.set_ylim(0, max(f1_matrix.values.flatten()) * 1.20)
ax.legend(title="Strategy", loc="upper left", bbox_to_anchor=(1.02, 1.0))
plt.xticks(rotation=0)
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=2, fontsize=8)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig2_f1_by_strategy.png", dpi=150, bbox_inches="tight")
plt.show()


# ------------------------------------------------------------------
# Figure 3: Operating-point trade-off — Recall vs FPR
# Shows that per-language tuning lifts recall, but at very different FPR cost per model
# ------------------------------------------------------------------
markers = {"default_0.5": "o", "global_recall@0.7": "s",
           "per_language_oracle": "D", "per_language_proper": "^"}
colors = {m: c for m, c in zip(MODEL_ORDER, sns.color_palette("Set1", n_colors=4))}

fig, ax = plt.subplots(figsize=(10, 7))
for model in MODEL_ORDER:
    sub = (overall_df[overall_df["model"] == model]
           .set_index("setting").reindex(SETTING_ORDER))
    ax.plot(sub["recall"], sub["fpr"], "-", color=colors[model], alpha=0.35, zorder=1)
    for setting in SETTING_ORDER:
        row = sub.loc[setting]
        ax.scatter(row["recall"], row["fpr"],
                   s=200, marker=markers[setting], color=colors[model],
                   edgecolor="black", linewidth=1, zorder=5)

ax.plot([0, 1], [0, 1], "--", color="grey", alpha=0.4)
ax.set_xlabel("Recall (multilingual val)", fontsize=11)
ax.set_ylabel("FPR (multilingual val)", fontsize=11)
ax.set_title("Operating-Point Trade-off: Recall vs FPR\n"
             "(lines connect strategies per model — closer to upper-left = better)",
             fontsize=12, fontweight="bold")
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)

model_handles = [mlines.Line2D([], [], color=colors[m], marker="o", linestyle="-",
                                markersize=10, markeredgecolor="black", label=m)
                 for m in MODEL_ORDER]
strat_handles = [mlines.Line2D([], [], color="grey", marker=markers[s], linestyle="",
                                markersize=10, markeredgecolor="black", label=SETTING_LABEL[s])
                 for s in SETTING_ORDER]
leg1 = ax.legend(handles=model_handles, title="Model",
                 loc="upper left", bbox_to_anchor=(1.02, 1.0))
ax.add_artist(leg1)
ax.legend(handles=strat_handles, title="Strategy",
          loc="upper left", bbox_to_anchor=(1.02, 0.60))
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig3_recall_vs_fpr.png", dpi=150, bbox_inches="tight")
plt.show()


# ------------------------------------------------------------------
# Figure 4: Per-language F1 heatmap under per_language_proper (D)
# Shows where each model struggles linguistically under the deployable strategy
# ------------------------------------------------------------------
heatmap_data = (per_lang_df[per_lang_df["setting"] == "per_language_proper"]
                .pivot_table(index="model", columns="lang", values="f1")
                .reindex(MODEL_ORDER))

fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(heatmap_data, annot=True, fmt=".3f", cmap="RdYlGn",
            cbar_kws={"label": "F1"},
            linewidths=0.5, linecolor="white", ax=ax,
            vmin=heatmap_data.min().min() * 0.9,
            vmax=heatmap_data.max().max() * 1.05)
ax.set_title("Per-Language F1 under Per-Language Proper Tuning (Strategy D)",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Language")
ax.set_ylabel("Model")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig4_per_language_f1_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


print(f"\nAll figures saved to: {FIG_DIR}")
for f in sorted(os.listdir(FIG_DIR)):
    print(f"  - {f}")

In [ ]:
# Module 6.5: Update project state
STATE_FILE = f"{PROJECT_ROOT}/project_state/run_state.json"

with open(STATE_FILE, "r") as f:
    state = json.load(f)

state["module_6_evaluation"] = {
    "status": "done",
    "settings_evaluated": [
        "default_0.5",
        "global_recall@0.7",
        "per_language_oracle",
        "per_language_proper",
    ],
    "metrics": [
        "roc_auc", "pr_auc",
        "precision", "recall", "f1",
        "fpr", "fnr",
        "positive_prediction_rate",
    ],
    "evaluation_subsets": {
        "default_0.5":         "full multilingual val",
        "global_recall@0.7":   "full multilingual val",
        "per_language_oracle": "full multilingual val",
        "per_language_proper": "val_eval (50% of val, leak-free)",
    },
    "output_dir": MODULE6_PATH,
    "next_step": "report_writeup",
}

with open(STATE_FILE, "w") as f:
    json.dump(state, f, indent=4)

print("Module 6 marked done in run_state.json.")
print("Next step:", state["module_6_evaluation"]["next_step"])

## Module 7: Test Performance

In [ ]:
!pip install -q transformers datasets accelerate

from google.colab import drive
drive.mount("/content/drive")

import os, glob
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

PROJECT_ROOT = "/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification"
OUTPUT_PATH = f"{PROJECT_ROOT}/module4_outputs"
MODULE5_PATH = f"{PROJECT_ROOT}/module5_outputs"
TEST_OUTPUT_PATH = f"{PROJECT_ROOT}/module5_test_outputs"
os.makedirs(TEST_OUTPUT_PATH, exist_ok=True)

RAW_TEST_PATH = f"{PROJECT_ROOT}/test.csv"

possible_label_paths = [
    f"{PROJECT_ROOT}/test_labels.csv",
    f"{PROJECT_ROOT}/test-labels.csv",
]
TEST_LABELS_PATH = next((p for p in possible_label_paths if os.path.exists(p)), None)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_TEST_PATH exists:", os.path.exists(RAW_TEST_PATH))
print("TEST_LABELS_PATH:", TEST_LABELS_PATH)
print("GPU available:", torch.cuda.is_available())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT: /content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification
RAW_TEST_PATH exists: True
TEST_LABELS_PATH: /content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/test_labels.csv
GPU available: True


In [ ]:
MODEL_NAME = "XLM-R"
PROB_COL = "pred_prob_xlmr"

def has_model_weights(path):
    return (
        os.path.exists(os.path.join(path, "model.safetensors"))
        or os.path.exists(os.path.join(path, "pytorch_model.bin"))
    )

candidate_dirs = [
    f"{OUTPUT_PATH}/xlmr_best_model",
    f"{OUTPUT_PATH}/xlmr_model",
]

candidate_dirs += sorted(
    glob.glob(f"{OUTPUT_PATH}/xlmr_model/checkpoint-*"),
    key=lambda p: os.path.getmtime(p),
    reverse=True,
)

print("Candidate model dirs:")
for d in candidate_dirs:
    print(d, "exists:", os.path.exists(d), "has_weights:", has_model_weights(d) if os.path.exists(d) else False)

MODEL_DIR = None
for d in candidate_dirs:
    if os.path.exists(d) and has_model_weights(d):
        MODEL_DIR = d
        break

if MODEL_DIR is None:
    raise FileNotFoundError("No XLM-R model weights found. Need model.safetensors or pytorch_model.bin.")

print("\nUsing MODEL_DIR:", MODEL_DIR)


Candidate model dirs:
/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/module4_outputs/xlmr_best_model exists: True has_weights: False
/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/module4_outputs/xlmr_model exists: True has_weights: False
/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/module4_outputs/xlmr_model/checkpoint-471960 exists: True has_weights: True
/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/module4_outputs/xlmr_model/checkpoint-235980 exists: True has_weights: True

Using MODEL_DIR: /content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/module4_outputs/xlmr_model/checkpoint-471960


In [ ]:
test_df = pd.read_csv(RAW_TEST_PATH).copy()

if "comment_text" not in test_df.columns:
    if "content" in test_df.columns:
        test_df = test_df.rename(columns={"content": "comment_text"})
    else:
        raise ValueError(f"Cannot find comment text column. Columns: {test_df.columns.tolist()}")

required = {"id", "comment_text", "lang"}
missing = required - set(test_df.columns)
if missing:
    raise ValueError(f"test.csv missing columns: {missing}")

test_df = test_df[["id", "comment_text", "lang"]].copy()
test_df["comment_text"] = test_df["comment_text"].fillna("").astype(str)
test_df["lang"] = test_df["lang"].astype(str)

print("Raw test shape:", test_df.shape)
print(test_df["lang"].value_counts())

tokenizer_dir = f"{OUTPUT_PATH}/xlmr_best_model"
if not os.path.exists(os.path.join(tokenizer_dir, "tokenizer.json")):
    tokenizer_dir = MODEL_DIR

tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)

if torch.cuda.is_available():
    model = model.to("cuda")

test_dataset = Dataset.from_pandas(test_df[["comment_text"]], preserve_index=False)

def tokenize_function(batch):
    return tokenizer(batch["comment_text"], truncation=True, max_length=128)

test_dataset = test_dataset.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(model=model, data_collator=data_collator)

pred = trainer.predict(test_dataset)
logits = pred.predictions

if logits.ndim == 2 and logits.shape[1] == 2:
    test_prob = F.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
else:
    test_prob = 1 / (1 + np.exp(-logits.reshape(-1)))

test_df[PROB_COL] = test_prob

print("Prediction done.")
print(test_df[[ "id", "lang", PROB_COL ]].head())
print(test_df[PROB_COL].describe())


Raw test shape: (63812, 3)
lang
tr    14000
pt    11012
ru    10948
fr    10920
it     8494
es     8438
Name: count, dtype: int64


Loading weights:   0%|          | 0/201 [00:02<?, ?it/s]

Map:   0%|          | 0/63812 [00:00<?, ? examples/s]

Prediction done.
   id lang  pred_prob_xlmr
0   0   tr        0.016317
1   1   ru        0.002025
2   2   it        0.020294
3   3   tr        0.003356
4   4   tr        0.002446
count    63812.000000
mean         0.074852
std          0.225301
min          0.001801
25%          0.002479
50%          0.007051
75%          0.025627
max          0.995215
Name: pred_prob_xlmr, dtype: float64


In [ ]:
d_thr = pd.read_csv(f"{MODULE5_PATH}/strategy_d_per_language_thresholds.csv")
ab_thr = pd.read_csv(f"{MODULE5_PATH}/strategy_ab_thresholds.csv")

thr_map = dict(
    d_thr[d_thr["model"] == MODEL_NAME]
    .set_index("lang")["threshold"]
    .astype(float)
)

fallback_row = ab_thr[
    (ab_thr["model"] == MODEL_NAME)
    & (ab_thr["strategy"] == "global_recall")
    & (ab_thr["target_recall"] == 0.7)
]

if len(fallback_row) != 1:
    raise ValueError(f"Expected one global threshold for {MODEL_NAME}, found {len(fallback_row)}")

fallback_thr = float(fallback_row["threshold"].iloc[0])

test_df["strategy_d_threshold"] = test_df["lang"].map(thr_map)
test_df["threshold_source"] = np.where(
    test_df["strategy_d_threshold"].isna(),
    "global_fallback",
    "per_language",
)
test_df["strategy_d_threshold"] = test_df["strategy_d_threshold"].fillna(fallback_thr)
test_df["strategy_d_pred"] = (test_df[PROB_COL] >= test_df["strategy_d_threshold"]).astype(int)

pred_path = f"{TEST_OUTPUT_PATH}/xlmr_strategy_d_original_test_predictions.csv"
submission_path = f"{TEST_OUTPUT_PATH}/xlmr_strategy_d_original_test_submission_labels.csv"

test_df.to_csv(pred_path, index=False)
test_df[["id", "strategy_d_pred"]].rename(columns={"strategy_d_pred": "toxic"}).to_csv(
    submission_path,
    index=False,
)

print("Saved prediction file:", pred_path)
print("Saved submission file:", submission_path)

print("\nShape:", test_df.shape)
print("\nPrediction counts:")
print(test_df["strategy_d_pred"].value_counts())
print("\nThreshold source counts:")
print(test_df["threshold_source"].value_counts())
print("\nPer-language positive rate:")
print(test_df.groupby("lang")["strategy_d_pred"].agg(["count", "mean"]))


Saved prediction file: /content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/module5_test_outputs/xlmr_strategy_d_original_test_predictions.csv
Saved submission file: /content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/module5_test_outputs/xlmr_strategy_d_original_test_submission_labels.csv

Shape: (63812, 7)

Prediction counts:
strategy_d_pred
0    48863
1    14949
Name: count, dtype: int64

Threshold source counts:
threshold_source
global_fallback    32880
per_language       30932
Name: count, dtype: int64

Per-language positive rate:
      count      mean
lang                 
es     8438  0.442048
fr    10920  0.196703
it     8494  0.360607
pt    11012  0.187704
ru    10948  0.160943
tr    14000  0.155643


In [ ]:
if TEST_LABELS_PATH is None:
    raise FileNotFoundError("Cannot find test_labels.csv or test-labels.csv under PROJECT_ROOT.")

labels = pd.read_csv(TEST_LABELS_PATH).copy()

print("Label columns:", labels.columns.tolist())
print("Label shape:", labels.shape)

if "toxic" not in labels.columns:
    raise ValueError(f"test labels missing toxic column. Columns: {labels.columns.tolist()}")

labels = labels[["id", "toxic"]].copy()

# Some datasets use -1 for unavailable labels.
labels = labels[labels["toxic"].isin([0, 1])].copy()
labels["toxic"] = labels["toxic"].astype(int)

eval_df = test_df.merge(labels, on="id", how="inner", suffixes=("", "_true"))

print("Eval rows after merging labels:", eval_df.shape)
print("Unmatched predictions:", len(test_df) - len(eval_df))

y_true = eval_df["toxic"].astype(int)
y_prob = eval_df[PROB_COL].astype(float)
y_pred = eval_df["strategy_d_pred"].astype(int)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

overall_metrics = {
    "model": MODEL_NAME,
    "setting": "strategy_d_on_original_test",
    "n": len(eval_df),
    "roc_auc": roc_auc_score(y_true, y_prob),
    "pr_auc": average_precision_score(y_true, y_prob),
    "precision": precision_score(y_true, y_pred, zero_division=0),
    "recall": recall_score(y_true, y_pred, zero_division=0),
    "f1": f1_score(y_true, y_pred, zero_division=0),
    "fpr": fp / (fp + tn) if (fp + tn) > 0 else np.nan,
    "fnr": fn / (fn + tp) if (fn + tp) > 0 else np.nan,
    "positive_prediction_rate": y_pred.mean(),
    "tp": tp,
    "fp": fp,
    "fn": fn,
    "tn": tn,
}

overall_df = pd.DataFrame([overall_metrics])
overall_metrics_path = f"{TEST_OUTPUT_PATH}/xlmr_strategy_d_original_test_overall_metrics.csv"
overall_df.to_csv(overall_metrics_path, index=False)

print("\nOverall test metrics:")
print(overall_df.round(4).to_string(index=False))

rows = []
for lang, g in eval_df.groupby("lang"):
    yt = g["toxic"].astype(int)
    yp = g[PROB_COL].astype(float)
    yh = g["strategy_d_pred"].astype(int)

    tn, fp, fn, tp = confusion_matrix(yt, yh, labels=[0, 1]).ravel()
    has_both = yt.nunique() > 1

    rows.append({
        "lang": lang,
        "n": len(g),
        "threshold_source_main": g["threshold_source"].mode().iloc[0],
        "roc_auc": roc_auc_score(yt, yp) if has_both else np.nan,
        "pr_auc": average_precision_score(yt, yp) if has_both else np.nan,
        "precision": precision_score(yt, yh, zero_division=0),
        "recall": recall_score(yt, yh, zero_division=0),
        "f1": f1_score(yt, yh, zero_division=0),
        "fpr": fp / (fp + tn) if (fp + tn) > 0 else np.nan,
        "fnr": fn / (fn + tp) if (fn + tp) > 0 else np.nan,
        "positive_prediction_rate": yh.mean(),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    })

per_lang_df = pd.DataFrame(rows).sort_values("lang")
per_lang_metrics_path = f"{TEST_OUTPUT_PATH}/xlmr_strategy_d_original_test_per_language_metrics.csv"
per_lang_df.to_csv(per_lang_metrics_path, index=False)

print("\nPer-language test metrics:")
print(per_lang_df.round(4).to_string(index=False))

print("\nSaved metrics:")
print(overall_metrics_path)
print(per_lang_metrics_path)


Label columns: ['id', 'toxic']
Label shape: (63812, 2)
Eval rows after merging labels: (63812, 8)
Unmatched predictions: 0

Overall test metrics:
model                     setting     n  roc_auc  pr_auc  precision  recall     f1    fpr    fnr  positive_prediction_rate   tp   fp   fn    tn
XLM-R strategy_d_on_original_test 63812   0.8264  0.5524     0.5349  0.5549 0.5447 0.1407 0.4451                    0.2343 7996 6953 6414 42449

Per-language test metrics:
lang     n threshold_source_main  roc_auc  pr_auc  precision  recall     f1    fpr    fnr  positive_prediction_rate   tp   fp   fn    tn
  es  8438          per_language   0.8279  0.7504     0.6684  0.7424 0.7034 0.2435 0.2576                    0.4420 2493 1237  865  3843
  fr 10920       global_fallback   0.7815  0.5929     0.6331  0.4072 0.4956 0.1040 0.5928                    0.1967 1360  788 1980  6792
  it  8494          per_language   0.7716  0.3898     0.3670  0.6866 0.4783 0.2828 0.3134                    0.3606 1124 1939  